# Import data into SQL Lite DB

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text
import os
from datetime import datetime
import re

# Konfiguration
photometer_name = 'stars926'
db_name = f'../data/{photometer_name}_data.db'
ecsv_folder = f'../python/TESS-IDA-TOOLS/jupyter/ECSV/{photometer_name}/'

# SQLite-Verbindung erstellen
engine = create_engine(f'sqlite:///{db_name}')

# Tabelle in SQLite erstellen (falls nicht vorhanden)
create_table_query = f'''
CREATE TABLE IF NOT EXISTS {photometer_name}_data (
    time TIMESTAMP,
    enclosure_temperature DOUBLE PRECISION,
    sky_temperature DOUBLE PRECISION,
    frequency DOUBLE PRECISION,
    msas DOUBLE PRECISION,
    zp DOUBLE PRECISION,
    sequence_number BIGINT,
    sun_alt DOUBLE PRECISION,
    moon_alt DOUBLE PRECISION,
    moon_illumination DOUBLE PRECISION
);
'''

# Tabelle erstellen
with engine.connect() as connection:
    connection.execute(text(create_table_query))

# Muster zur Überprüfung des Dateinamensformats
filename_pattern = re.compile(rf"{photometer_name}_\d{{4}}-\d{{2}}\.ecsv")

# Importieren und Filtern der ECSV-Daten
for file_name in os.listdir(ecsv_folder):
    # Überprüfen, ob der Dateiname auf ".ecsv" endet und dem Format entspricht
    if file_name.endswith('.ecsv') and filename_pattern.match(file_name):
        ecsv_file = os.path.join(ecsv_folder, file_name)
        data = pd.read_csv(ecsv_file, comment='#', delimiter=',')

        # Spaltennamen anpassen
        data.columns = [
            'time', 
            'enclosure_temperature', 
            'sky_temperature', 
            'frequency', 
            'msas', 
            'zp', 
            'sequence_number', 
            'sun_alt', 
            'moon_alt', 
            'moon_illumination'
        ]

        # Gefilterte Daten in die SQLite-Datenbank importieren
        data.to_sql(f'{photometer_name}_data', engine, if_exists='append', index=False)
        print(f"Gefilterte Daten von {file_name} wurden erfolgreich importiert.")
    else:
        print(f"Datei {file_name} entspricht nicht dem erwarteten Format und wird übersprungen.")

print("Alle gefilterten Dateien wurden in die SQLite-Datenbank importiert.")


## Check if data is missing in the time span

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text
from datetime import datetime, timedelta

# Database connection settings for SQLite
db_name = 'stars289_data.db'

# Create an SQLite engine
engine = create_engine(f'sqlite:///{db_name}')

# Define the time range (you can adjust these dates)
start_date = datetime(2024, 1, 1)  # Start date of the range
end_date = datetime(2024, 12, 31)  # End date of the range

# Generate a list of all days within the time range
all_days = pd.date_range(start=start_date, end=end_date, freq='D')

# Query the database for each day and check if at least one entry exists
missing_days = []

with engine.connect() as connection:
    for day in all_days:
        # Check if there is at least one entry for the current day
        query = text(f'''
        SELECT 1 FROM stars289_data
        WHERE DATE(time) = '{day.date()}'
        LIMIT 1;
        ''')
        result = connection.execute(query).fetchone()

        if result is None:
            missing_days.append(day.date())  # Add to missing days if no entry found

# Output the missing days
if missing_days:
    print("The following days are missing from the database:")
    for missing_day in missing_days:
        print(missing_day)
else:
    print("No missing days within the given time span.")
